**Estimator Demo — Running on IBM Quantum Hardware.**

This notebook demonstrates how to build a Bell state circuit, transpile it
for a real quantum processor, submit it to IBM Quantum via the cloud runtime,
and retrieve expectation values of several Pauli observables.
It introduces several Qiskit Runtime concepts that are worth understanding
before running on real hardware for the first time.

---
**Cell 01 — Build the Bell state circuit.**

A Bell state is the simplest maximally entangled two-qubit state.
Starting from $|00\rangle$, a Hadamard gate on $q_0$ creates the
superposition $\frac{1}{\sqrt{2}}(|0\rangle+|1\rangle)$, and a
CNOT gate (controlled-X, `cx`) entangles the two qubits:

$$|00\rangle
\xrightarrow{H\otimes I}
\frac{1}{\sqrt{2}}(|0\rangle+|1\rangle)|0\rangle
\xrightarrow{\text{CNOT}}
\frac{1}{\sqrt{2}}(|00\rangle+|11\rangle)$$

This is an idealized *logical* circuit — it assumes perfect gates and
arbitrary qubit connectivity.
Real hardware has neither, which is why we must transpile before running.

In [ ]:
"""estimator_demo_hardware.ipynb"""

# Cell 01 - Create a Bell State (|00> + |11>)

from qiskit import QuantumCircuit

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)

qc.draw(output="mpl")

---
**Cell 02 — Transpile and submit to hardware.**

**QiskitRuntimeService.**
This is Qiskit's client for IBM's cloud-hosted quantum runtime.
Calling `QiskitRuntimeService(channel="ibm_cloud")` loads the saved
credentials from `~/.qiskit/qiskit-ibm.json` (written by
`qiskit_save_credential.ipynb`) and opens an authenticated session.
`least_busy()` queries all backends on your account and returns the
one with the shortest current job queue.

**Pass manager and transpiler.**
A *transpiler* converts a logical circuit into an *instruction set
architecture* (ISA) circuit — one that uses only the native gates
of the target device and respects its qubit connectivity graph.
For example, a real device may have no direct connection between $q_0$
and $q_3$, so a CNOT between them must be decomposed into a sequence
of SWAP and native-gate operations.
A *pass manager* is the pipeline that applies these transformations.
`generate_preset_pass_manager(optimization_level=1)` gives a balanced
pipeline: more optimization than level 0 (none) but faster compilation
than levels 2–3.
The result `isa_circuit` is what actually runs on the chip.

**SparsePauliOp.**
On a quantum computer we cannot directly read out a state vector —
we can only measure qubits in the computational ($Z$) basis.
To measure *other* physical quantities we express them as sums of
Pauli operators ($I$, $X$, $Y$, $Z$) acting on qubits.
`SparsePauliOp` represents such an operator efficiently by storing
only the non-identity terms.
For example, `SparsePauliOp("ZZ")` represents the operator
$Z\otimes Z$, which measures whether the two qubits are correlated
in the $Z$ basis.
The six observables here — `IZ`, `IX`, `ZI`, `XI`, `ZZ`, `XX` —
probe all single-qubit and two-qubit Pauli correlations of the Bell state.

**Mapped observables.**
Transpilation reorders and remaps qubits onto physical hardware qubits.
`observable.apply_layout(isa_circuit.layout)` updates each observable
to track that remapping, so that `IZ` on logical qubits 0 and 1 becomes
the correct operator on whichever physical qubits the transpiler chose.
Without this step the Estimator would measure the wrong qubits.

**EstimatorV2.**
The *Estimator* is a Qiskit Runtime *primitive* — a high-level abstraction
that handles the full workflow of submitting shots, collecting counts,
and computing expectation values $\langle O \rangle = \langle\psi|O|\psi\rangle$
for each observable $O$.
You do not need to manually convert counts to expectation values;
the Estimator does it for you.
`default_shots = 5000` sets the number of circuit executions used to
estimate each expectation value — more shots reduce statistical noise.

**Resilience level.**
`resilience_level = 1` enables *readout error mitigation*, the lowest
level of built-in error suppression.
Real hardware has imperfect measurement: a qubit in state $|1\rangle$
is occasionally read as 0, and vice versa.
At level 1, the runtime runs additional calibration circuits to
characterize these errors and correct the expectation values
statistically.
Level 0 disables all mitigation; levels 2 and 3 add gate-level
noise suppression at increasing computational cost.

**Job submission.**
`estimator.run([(isa_circuit, mapped_observables)])` packages the
circuit and its observables into a *PUB* (Primitive Unified Bloc) —
the unit of work the Estimator accepts — and submits it to the
cloud queue.
The call returns immediately with a `job` handle; the circuit has
been queued but has not yet run.
The job ID printed at the end can be used to retrieve results later
even if this notebook session ends before the job completes.

In [ ]:
# Cell 02 - Run circuit on a target backend

from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_ibm_runtime import QiskitRuntimeService

# Connect to IBM Cloud using credentials in ~/.qiskit/qiskit-ibm.json
service = QiskitRuntimeService(channel="ibm_cloud")

# Select the least busy backend that is operational
backend = service.least_busy(simulator=False, operational=True)
config = backend.configuration()
print(
    f"{config.backend_name:15}: Qubits = {config.n_qubits}: Gates = {config.basis_gates}"
)

# Transpile the quantum circuit for the target backend
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

# Display how the circuit will actually run on this backend
display(isa_circuit.draw("mpl", idle_wires=False))

estimator = Estimator(backend)
estimator.options.default_shots = 5000
estimator.options.resilience_level = 1

observables_labels = ["IZ", "IX", "ZI", "XI", "ZZ", "XX"]
observables = [SparsePauliOp(label) for label in observables_labels]
mapped_observables = [
    observable.apply_layout(isa_circuit.layout) for observable in observables
]

job = estimator.run([(isa_circuit, mapped_observables)])

print(f"Job ID: {job.job_id()}")

---
**Cell 03 — Retrieve results and plot.**

**job.result().**
`job.result()` is a *blocking* call. This makes your code wait until the job has
finished running on the hardware and all results have been returned
from the cloud.
Depending on the queue, this can take anywhere from seconds to hours.
The return value is a list of *PUB results*, one per PUB submitted.

**PUB and pub_result.**
A *PUB* (Primitive Unified Bloc) is the Qiskit Runtime term for a
single (circuit, observables) pair submitted to a primitive.
Because `estimator.run()` accepts a list of PUBs, `job.result()` returns
a list of corresponding results.
Here we submitted one PUB, so `job.result()[0]` extracts the single
result for that PUB.

**pub_result.data.evs.**
`evs` (expectation values) is a NumPy array containing one estimated
expectation value $\langle O_i \rangle$ for each observable $O_i$
in `mapped_observables`.
For the ideal Bell state $\frac{1}{\sqrt{2}}(|00\rangle+|11\rangle)$
the theoretical values are:
- $\langle IZ \rangle = 0$, $\langle ZI \rangle = 0$ -- neither qubit has a definite $Z$ value individually
- $\langle IX \rangle = 0$, $\langle XI \rangle = 0$ -- same for $X$
- $\langle ZZ \rangle = 1$ -- the qubits are perfectly correlated in $Z$
- $\langle XX \rangle = 1$ -- the qubits are perfectly correlated in $X$

**Why does $\langle IZ \rangle = 0$ indicate qubit 0 has no definite $Z$ value?**\
The operator $IZ$ acts as the identity on qubit 1 and as $Z$ on qubit 0.
It asks: ignoring qubit 1 entirely, does qubit 0 have a definite value
in the $Z$ basis?
The $Z$ operator assigns eigenvalue $+1$ to $|0\rangle$ and $-1$ to $|1\rangle$.
In the Bell state the $|00\rangle$ and $|11\rangle$ terms each have
amplitude $\frac{1}{\sqrt{2}}$, so qubit 0 is equally likely to be
measured as 0 or 1:

$$\langle IZ \rangle = (+1)\cdot P(q_0 = 0) + (-1)\cdot P(q_0 = 1) = \tfrac{1}{2}(+1) + \tfrac{1}{2}(-1) = 0$$

Qubit 0 is in a superposition of $|0\rangle$ and $|1\rangle$ with equal
weight -- it has no definite $Z$ value on its own.
The same argument gives $\langle ZI \rangle = 0$ for qubit 1.

**Why does $\langle ZZ \rangle = 1$ indicate perfect $Z$-basis correlation?**\
The $ZZ$ operator assigns eigenvalue $+1$ when both qubits are the same
($|00\rangle$ or $|11\rangle$) and $-1$ when they differ
($|01\rangle$ or $|10\rangle$).
The Bell state contains *only* the same-parity terms with zero amplitude
on $|01\rangle$ and $|10\rangle$, so every shot returns 00 or 11, never 01 or 10:

$$\langle ZZ \rangle = (+1)\cdot P(00) + (+1)\cdot P(11) + (-1)\cdot P(01) + (-1)\cdot P(10) = \tfrac{1}{2} + \tfrac{1}{2} + 0 + 0 = 1$$

Neither qubit has a definite value alone, yet they are always the same
as each other. This is the hallmark of entanglement.

**Why does $\langle XX \rangle = 1$ indicate perfect $X$-basis correlation?**\
The $X$ basis uses the states $|{+}\rangle = \frac{1}{\sqrt{2}}(|0\rangle+|1\rangle)$
and $|{-}\rangle = \frac{1}{\sqrt{2}}(|0\rangle-|1\rangle)$, with
eigenvalues $+1$ and $-1$ respectively.
Rewriting $|0\rangle$ and $|1\rangle$ in terms of $|{+}\rangle$ and
$|{-}\rangle$ and substituting into the Bell state:

$$\frac{1}{\sqrt{2}}(|00\rangle+|11\rangle) = \frac{1}{\sqrt{2}}(|{+}{+}\rangle + |{-}{-}\rangle)$$

The Bell state has exactly the same form in the $X$ basis as in the
$Z$ basis -- only same-parity terms, no cross terms.
The $XX$ operator gives $+1$ on $|{+}{+}\rangle$ and $|{-}{-}\rangle$,
so $\langle XX \rangle = 1$ by the same calculation as $\langle ZZ \rangle$.
This simultaneous perfect correlation in *both* the $Z$ and $X$ bases
cannot be reproduced by any classical pair of correlated bits: a classical
pair that always agrees in $Z$ (e.g. both are always 0) would give
$\langle XX \rangle = 0$, not 1.

Deviations from these ideal values in the plot reflect hardware noise.

**pub_result.data.stds.**
`stds` is a NumPy array of standard deviations (error bars) for each
expectation value, estimated from the shot statistics.
They are included in the plot to show the statistical uncertainty
in each measurement.
Larger shot counts reduce these error bars as $1/\sqrt{N_\text{shots}}$.

In [ ]:
# Cell 03 - Wait for job to complete and plot the results

from matplotlib import pyplot as plt

# job.result() blocks until the hardware job finishes (may take minutes)
pub_result = job.result()[0]

values = pub_result.data.evs  # expectation values for each observable
errors = pub_result.data.stds  # standard deviations (shot-noise error bars)

plt.errorbar(
    observables_labels,
    values,
    yerr=errors,
    fmt="-o",
    capsize=4,
    color="steelblue",
    ecolor="tomato",
    capthick=1.5,
    elinewidth=1.5,
)
plt.axhline(0, color="gray", linewidth=0.8, linestyle="--")
plt.xlabel("Observable")
plt.ylabel(r"Expectation value $\langle O \rangle$")
plt.title("Bell State Observables (IBM Hardware)")
plt.ylim(-1.2, 1.2)
plt.tight_layout()
plt.show()
